This notebook copies the content from https://www.youtube.com/watch?v=Eug2clsLtFs <br>

* ReAct agent: langchain.AgentType.REACT_DOCSTORE <br>
* LLM: OpenAI <br>
* RAG:
    * langchain.vectorstores.FAISS and
    * langchain.text_splitte.RecursiveCharacterTextSplitter

In [1]:
!pip -q install langchain huggingface_hub openai langchain-community pypdf PyMuPDF sentence-transformers faiss-cpu

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.4/50.4 kB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 378.9/378.9 kB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 33.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.4/76.4 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.9/77.9 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 318.9/318.9 kB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 399.9/399.9 kB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 292.2/292.2 kB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.3/49.3 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.9/141.9 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.3/58.3 kB 2.8 MB/s eta 0:00:00


In [14]:
from langchain.schema import Document  # Import the Document class
import os
from langchain import OpenAI
from langchain.agents import initialize_agent, Tool
from langchain.agents import AgentType
from langchain.vectorstores import FAISS
from langchain.embeddings import HuggingFaceEmbeddings
import fitz  # For PDF extraction
from langchain.text_splitter import RecursiveCharacterTextSplitter
import numpy as np
import faiss
import openai

In [32]:


# Set your OpenAI API Key
os.environ["OPENAI_API_KEY"] = ""



model_name = 'all-MiniLM-L6-v2'  # You can change this model if desired
embedding_model = HuggingFaceEmbeddings(model_name=model_name)


def extract_text_from_pdf(pdf_path):
    pdf_document = fitz.open(pdf_path)
    text = ""
    for page_num in range(pdf_document.page_count):
        page = pdf_document.load_page(page_num)
        text += page.get_text()
    return text


def chunk_document(document_text):
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=20)
    return text_splitter.split_text(document_text)


def generate_documents(chunks):
    # Convert text chunks to Document objects
    documents = [Document(page_content=chunk) for chunk in chunks]  # Wrap each chunk in a Document object
    return documents  # Return both embeddings and the Document objects


def create_faiss_index(documents, embeddings):
    faiss_index = FAISS.from_documents(embedding=embedding_model, documents=documents)  # Use both embeddings and documents
    return faiss_index


def faiss_similarity_search(query, vectorstore, top_k=2):

    query_embedding = embedding_model.embed_query(query)
    results = vectorstore.similarity_search_by_vector(query_embedding, k=top_k)
    return results

def faiss_search(query):
    return faiss_similarity_search(query, faiss_index)


/usr/local/lib/python3.10/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [33]:
pdf_path = "/content/naf23.pdf"
pdf_text = extract_text_from_pdf(pdf_path)
chunks = chunk_document(pdf_text)
documents = generate_documents(chunks)
faiss_index = create_faiss_index(documents, embedding_model)

In [34]:
tools = [
    Tool(
        name="Search",
        func=faiss_search,
        description="search documents"
    ),
    Tool(
        name="Lookup",
        func=faiss_search,
        description="search documents"
    )
]
llm = OpenAI(temperature=0.1, model_name="gpt-3.5-turbo-instruct")
react_agent = initialize_agent(tools, llm, agent=AgentType.REACT_DOCSTORE, verbose=True,handle_parsing_errors=True)



In [29]:
question = "Can you analyse  National Ataxia Foudnations's current assets, provide potential drivers"
react_agent.run(question)



> Entering new AgentExecutor chain...
Thought: I need to search National Ataxia Foundation, find its current assets, and provide potential drivers for those assets.
Action: Search[National Ataxia Foundation]
Observation: [Document(metadata={}, page_content='with ataxia, family members and health care professionals on a “direct request” basis through the NAF office. The \nFoundation’s advocacy activities, such as Hill Days, serve to raise awareness about ataxia and policy issues \nimpacting the ataxia community. The annual ataxia conference, and sponsorship and/or participation in national and \ninternational meetings, symposiums and seminars also serve to increase ataxia awareness. \n \nThe National Ataxia Foundation attempts to locate people and families with hereditary and sporadic ataxia in order to \nprovide them with information about ataxia. This is accomplished by providing updated information about ataxia to \nthe general public by maintaining a website, social media channels

'donations, in-kind contributions, conference income, earned income, investment income'

In [35]:
question = "National Ataxia Foudnations's total expense increased from 4,131,189 in 2022 to 4,184,787 in 2023, do you know why, what caused it"
react_agent.run(question)



> Entering new AgentExecutor chain...
Thought: I need to search National Ataxia Foundation, find their expenses for 2022 and 2023, and determine the reason for the increase.
Action: Search[National Ataxia Foundation]
Observation: [Document(metadata={}, page_content='The National Ataxia Foundation attempts to locate people and families with hereditary and sporadic ataxia in order to \nprovide them with information about ataxia. This is accomplished by providing updated information about ataxia to \nthe general public by maintaining a website, social media channels, newsletters, and webinars. The Foundation'), Document(metadata={}, page_content='Foundation’s mission is to accelerate the development of treatments and a cure while working to improve the lives of \nthose living with Ataxia. \n \nTo further the mission, the Foundation provides services in the following program areas: \n \nEducation and Service \n \nIncreased awareness about hereditary and sporadic ataxia disorders is promo

'increase in expenses due to increase in program services and support and revenue'

Notes:

* this is just some exploration, there might be some errors in selecting the correct packages, techniques, etc.

Observations:
* 'compare' not implemented `"Compare[expenses 2022 2023]"`
* `support and revenue` has nothing to do with expenses
* this is just to show how model is thinking
    * search `NAF`
    * search `NAF's expense`
    * search `financial statements of NAF`
    * compare results
    * finish

* based on some of my testing that are not shown in this notebook, below factors are really important for the final quality of ReAct prompting
    * the llm model and its temperature
    * the quality of parsed pdf, how tables are parsed
    * maybe the size of chunk, whether all relevant information is captured in the chunk
    * maybe the embedding model